In [ ]:
# Student-specific inputs
LAST_NAME = "Magalong"
STUDENT_ID = "TUPM-26-1134"
SEED_NUM = int(STUDENT_ID[-1])
FAVORITE_ARTIST = "TAYLOR SWIFT"

# Character positions of FAVORITE_ARTIST (A=1, B=2, ... Z=26; spaces ignored)
ARTIST_POS = [ord(ch.upper()) - 64 for ch in FAVORITE_ARTIST if ch.isalpha()]

print("LAST_NAME      :", LAST_NAME)
print("SEED_NUM       :", SEED_NUM)
print("FAVORITE_ARTIST:", FAVORITE_ARTIST)
print("ARTIST_POS     :", ARTIST_POS)

LAST_NAME      : Magalong
SEED_NUM       : 4
FAVORITE_ARTIST: TAYLOR SWIFT
ARTIST_POS     : [20, 1, 25, 12, 15, 18, 19, 23, 9, 6, 20]


In [2]:
from collections import defaultdict, Counter

sources = [f"{LAST_NAME}_T{i}" for i in range(1, 6)]
categories = ["TEMPERATURE", "VOLTAGE", "CURRENT", "PRESSURE", "SPEED"]

def next_value(k):
    """Deterministic measurement value from SEED_NUM and artist character positions."""
    return SEED_NUM * ARTIST_POS[k % len(ARTIST_POS)] + k

records = []   # list of (source, category, value) tuples
k = 0
for i, src in enumerate(sources):
    for j in range(3):                       # 3 categories per source
        records.append((src, categories[(i + j) % 5], next_value(k)))
        k += 1

# Additional data: multiple measurements under the same category (T1 / TEMPERATURE)
for _ in range(2):
    records.append((sources[0], "TEMPERATURE", next_value(k)))
    k += 1

# Repeated record: exact duplicate of an existing test-source/category record
records.append(records[4])

print(f"{'#':<3}{'SOURCE':<14}{'CATEGORY':<14}VALUE")
for n, (s, c, v) in enumerate(records, 1):
    print(f"{n:<3}{s:<14}{c:<14}{v}")

#  SOURCE        CATEGORY      VALUE
1  Magalong_T1   TEMPERATURE   80
2  Magalong_T1   VOLTAGE       5
3  Magalong_T1   CURRENT       102
4  Magalong_T2   VOLTAGE       51
5  Magalong_T2   CURRENT       64
6  Magalong_T2   PRESSURE      77
7  Magalong_T3   CURRENT       82
8  Magalong_T3   PRESSURE      99
9  Magalong_T3   SPEED         44
10 Magalong_T4   PRESSURE      33
11 Magalong_T4   SPEED         90
12 Magalong_T4   TEMPERATURE   91
13 Magalong_T5   SPEED         16
14 Magalong_T5   TEMPERATURE   113
15 Magalong_T5   VOLTAGE       62
16 Magalong_T1   TEMPERATURE   75
17 Magalong_T1   TEMPERATURE   88
18 Magalong_T2   CURRENT       64


In [3]:
organized = defaultdict(lambda: defaultdict(list))
for s, c, v in records:
    organized[s][c].append(v)

for s in sorted(organized):
    print(s)
    for c in sorted(organized[s]):
        print(f"   {c:<12}: {organized[s][c]}")

Magalong_T1
   CURRENT     : [102]
   TEMPERATURE : [80, 75, 88]
   VOLTAGE     : [5]
Magalong_T2
   CURRENT     : [64, 64]
   PRESSURE    : [77]
   VOLTAGE     : [51]
Magalong_T3
   CURRENT     : [82]
   PRESSURE    : [99]
   SPEED       : [44]
Magalong_T4
   PRESSURE    : [33]
   SPEED       : [90]
   TEMPERATURE : [91]
Magalong_T5
   SPEED       : [16]
   TEMPERATURE : [113]
   VOLTAGE     : [62]


In [4]:
counts = Counter(records)
repeated = {rec: n for rec, n in counts.items() if n > 1}
distinct_records = set(records)
distinct_sources = {r[0] for r in records}
distinct_categories = {r[1] for r in records}
multi_measure = {(s, c): vals for s in organized for c, vals in organized[s].items() if len(vals) > 1}

print("Total records       :", len(records))
print("Distinct records    :", len(distinct_records))
print("Repeated records    :", repeated)
print("Distinct sources    :", sorted(distinct_sources))
print("Distinct categories :", sorted(distinct_categories))
print("Source/categories with multiple measurements:")
for key, vals in multi_measure.items():
    print("  ", key, "->", vals)

Total records       : 18
Distinct records    : 17
Repeated records    : {('Magalong_T2', 'CURRENT', 64): 2}
Distinct sources    : ['Magalong_T1', 'Magalong_T2', 'Magalong_T3', 'Magalong_T4', 'Magalong_T5']
Distinct categories : ['CURRENT', 'PRESSURE', 'SPEED', 'TEMPERATURE', 'VOLTAGE']
Source/categories with multiple measurements:
   ('Magalong_T1', 'TEMPERATURE') -> [80, 75, 88]
   ('Magalong_T2', 'CURRENT') -> [64, 64]


In [5]:
def stats(values):
    return {"count": len(values), "sum": sum(values), "min": min(values),
            "max": max(values), "avg": round(sum(values) / len(values), 2)}

by_category = defaultdict(list)
by_source = defaultdict(list)
for s, c, v in sorted(distinct_records):
    by_category[c].append(v)
    by_source[s].append(v)

category_summary = {c: stats(v) for c, v in by_category.items()}
source_summary = {s: stats(v) for s, v in by_source.items()}

print("--- By Category ---")
for c, st in category_summary.items():
    print(f"{c:<12}{st}")
print("\n--- By Source ---")
for s, st in source_summary.items():
    print(f"{s:<14}{st}")

--- By Category ---
CURRENT     {'count': 3, 'sum': 248, 'min': 64, 'max': 102, 'avg': 82.67}
TEMPERATURE {'count': 5, 'sum': 447, 'min': 75, 'max': 113, 'avg': 89.4}
VOLTAGE     {'count': 3, 'sum': 118, 'min': 5, 'max': 62, 'avg': 39.33}
PRESSURE    {'count': 3, 'sum': 209, 'min': 33, 'max': 99, 'avg': 69.67}
SPEED       {'count': 3, 'sum': 150, 'min': 16, 'max': 90, 'avg': 50.0}

--- By Source ---
Magalong_T1   {'count': 5, 'sum': 350, 'min': 5, 'max': 102, 'avg': 70.0}
Magalong_T2   {'count': 3, 'sum': 192, 'min': 51, 'max': 77, 'avg': 64.0}
Magalong_T3   {'count': 3, 'sum': 225, 'min': 44, 'max': 99, 'avg': 75.0}
Magalong_T4   {'count': 3, 'sum': 214, 'min': 33, 'max': 91, 'avg': 71.33}
Magalong_T5   {'count': 3, 'sum': 191, 'min': 16, 'max': 113, 'avg': 63.67}


In [6]:
ordered_records = sorted(distinct_records, key=lambda r: (r[0], r[1], r[2]))
categories_by_avg = sorted(category_summary.items(), key=lambda kv: kv[1]["avg"], reverse=True)
sources_by_total = sorted(source_summary.items(), key=lambda kv: kv[1]["sum"], reverse=True)
top_values = sorted(distinct_records, key=lambda r: r[2], reverse=True)[:5]

print("Categories ranked by average value:")
for rank, (c, st) in enumerate(categories_by_avg, 1):
    print(f"  {rank}. {c:<12} avg={st['avg']}")
print("\nSources ranked by total value:")
for rank, (s, st) in enumerate(sources_by_total, 1):
    print(f"  {rank}. {s:<14} sum={st['sum']}")
print("\nTop 5 measurements:")
for r in top_values:
    print("  ", r)

Categories ranked by average value:
  1. TEMPERATURE  avg=89.4
  2. CURRENT      avg=82.67
  3. PRESSURE     avg=69.67
  4. SPEED        avg=50.0
  5. VOLTAGE      avg=39.33

Sources ranked by total value:
  1. Magalong_T1    sum=350
  2. Magalong_T3    sum=225
  3. Magalong_T4    sum=214
  4. Magalong_T2    sum=192
  5. Magalong_T5    sum=191

Top 5 measurements:
   ('Magalong_T5', 'TEMPERATURE', 113)
   ('Magalong_T1', 'CURRENT', 102)
   ('Magalong_T3', 'PRESSURE', 99)
   ('Magalong_T4', 'TEMPERATURE', 91)
   ('Magalong_T4', 'SPEED', 90)


In [7]:
line = "=" * 64
print(line)
print("FINAL TEST DATA REPORT".center(64))
print(f"Operator: {LAST_NAME} | ID: {STUDENT_ID} | Seed: {SEED_NUM}".center(64))
print(line)
print(f"Total records: {len(records)}   Distinct: {len(distinct_records)}   Repeated: {len(repeated)}")
print(f"Sources: {len(distinct_sources)}   Categories: {len(distinct_categories)}\n")

print("ORGANIZED DATA (sorted)")
print(f"{'SOURCE':<14}{'CATEGORY':<14}{'VALUES'}")
for s in sorted(organized):
    for c in sorted(organized[s]):
        print(f"{s:<14}{c:<14}{organized[s][c]}")

print("\nCATEGORY SUMMARY (ranked by average)")
print(f"{'CATEGORY':<14}{'COUNT':>6}{'SUM':>8}{'MIN':>6}{'MAX':>6}{'AVG':>9}")
for c, st in categories_by_avg:
    print(f"{c:<14}{st['count']:>6}{st['sum']:>8}{st['min']:>6}{st['max']:>6}{st['avg']:>9}")

print("\nSOURCE SUMMARY (ranked by total)")
print(f"{'SOURCE':<14}{'COUNT':>6}{'SUM':>8}{'MIN':>6}{'MAX':>6}{'AVG':>9}")
for s, st in sources_by_total:
    print(f"{s:<14}{st['count']:>6}{st['sum']:>8}{st['min']:>6}{st['max']:>6}{st['avg']:>9}")

print("\nREPEATED RECORDS:", repeated)
print(line)

                     FINAL TEST DATA REPORT                     
        Operator: Magalong | ID: TUPM-26-1134 | Seed: 4         
Total records: 18   Distinct: 17   Repeated: 1
Sources: 5   Categories: 5

ORGANIZED DATA (sorted)
SOURCE        CATEGORY      VALUES
Magalong_T1   CURRENT       [102]
Magalong_T1   TEMPERATURE   [80, 75, 88]
Magalong_T1   VOLTAGE       [5]
Magalong_T2   CURRENT       [64, 64]
Magalong_T2   PRESSURE      [77]
Magalong_T2   VOLTAGE       [51]
Magalong_T3   CURRENT       [82]
Magalong_T3   PRESSURE      [99]
Magalong_T3   SPEED         [44]
Magalong_T4   PRESSURE      [33]
Magalong_T4   SPEED         [90]
Magalong_T4   TEMPERATURE   [91]
Magalong_T5   SPEED         [16]
Magalong_T5   TEMPERATURE   [113]
Magalong_T5   VOLTAGE       [62]

CATEGORY SUMMARY (ranked by average)
CATEGORY       COUNT     SUM   MIN   MAX      AVG
TEMPERATURE        5     447    75   113     89.4
CURRENT            3     248    64   102    82.67
PRESSURE           3     209    33    99